# Railroad Tycoon - Watch a Full Game

Minimal example: create the environment, run a full game with a simple scripted player, and
print what happens each turn. This is a faithful port of the CodinGame Summer Challenge 2026
referee - see `verify_rules.py` for the rule-by-rule checks against the original Java source.

For a live pygame window instead of printed output, use `docs/play_with_rendering.py` (a plain
script - pygame windows and Jupyter don't mix well).

## Create the environment

In [ ]:
from railroad_env import RailroadGymEnv

env = RailroadGymEnv(
    grid_height=None,           # None = the authentic random size (h 14..20, w = round(h*1.5))
    max_turns=100,
    opponent_strategy="random", # "passive" | "boss" | "random" | "greedy"
    seed=42,
)

obs, info = env.reset()
gs = env.game_state
print(f"Map {gs.width}x{gs.height}, {len(gs.towns)} towns, {len(gs.zones)} regions")
print(f"Observation shape: {obs.shape}  (height, width, channels)")


## The map

Towns want to be connected to specific *other* towns (`desired_connections`). Those wishes are
one-directional: if town 0 wants town 1, town 1 will not also want town 0. You score by owning
track that sits on the shortest built path between such a pair.

In [ ]:
print(f"{'id':>3}  {'(x, y)':>8}  desired connections")
for town in gs.towns:
    targets = ", ".join(str(o.id) for o in town.desired_connections) or "(none)"
    print(f"{town.id:>3}  ({town.coord[0]:>2}, {town.coord[1]:>2})  -> {targets}")

terrain_names = {0: "plains (cost 1)", 1: "river (cost 2)", 2: "mountain (cost 3)"}
import numpy as np
counts = dict(zip(*np.unique(gs.terrain, return_counts=True)))
print()
for t, name in terrain_names.items():
    print(f"{name:20s}: {counts.get(t, 0)} cells")


## A simple strategy for player 0

Each turn you get 3 paint points and 1 disruption point, neither of which carries over. The
`AUTOPLACE` action does the routing for you: it expands into the cheapest chain of
`PLACE_TRACKS` between two points that the budget allows.

In [ ]:
def player_strategy(game_state):
    """Work through the desired connections that aren't active yet."""
    for town in game_state.towns:
        for other in town.desired_connections:
            if other.id not in town.paths:
                return [("AUTOPLACE", town.coord[0], town.coord[1], other.coord[0], other.coord[1])]
    return [("WAIT",)]


## Run the game

In [ ]:
while True:
    actions = player_strategy(env.game_state)
    obs, reward, terminated, truncated, info = env.step({"actions": actions})

    if reward or info["turn"] % 20 == 0:
        placed = info["player_actions"]["placed"]
        print(
            f"Turn {info['turn']:3d} | Score {info['scores'][0]:5d}-{info['scores'][1]:<5d} "
            f"| reward {reward:+.0f} | placed {len(placed)} | opponent placed "
            f"{len(info['opponent_actions']['placed'])}"
        )

    if terminated:
        print(f"\nGame finished at turn {info['turn']}.")
        break

scores = info["scores"]
winner = "Player" if scores[0] > scores[1] else ("Opponent" if scores[1] > scores[0] else "Tie")
print(f"Final score - Player: {scores[0]}, Opponent: {scores[1]} ({winner})")


## Active connections

A connection is active once an unbroken path of track (or towns) links the two towns. Every turn
it stays active, each player earns 1 point for every track *they own* on that path - so a long
shared corridor pays both sides, and a neutral (contested) track pays neither.

In [ ]:
active = env.game_state.calculate_active_connections()
print(f"{len(active)} cells are part of at least one active connection")
for town in env.game_state.towns:
    for to_id, path in town.paths.items():
        owners = [env.game_state.tracks[y, x] for (x, y) in path]
        mine = sum(1 for o in owners if o == 0)
        theirs = sum(1 for o in owners if o == 1)
        print(f"  town {town.id} -> {to_id}: path of {len(path)} cells, "
              f"{mine} mine / {theirs} theirs per turn")
